# 311 Intake Assistant — Part C
**IE 5374 · Module 7 Hackathon · Topic 2: Staffing Boston's 311 City Service Teams**

A resident describes a problem in plain language. The assistant retrieves the closest of the **170 real 2024 request types**, picks one with **structured output**, looks up the **department and queue** from the data, and asks **one follow-up question** when the message is unclear.

```
resident message
   │
   ▼
① RAG retrieval ── FAISS over 170 "type cards" (OpenAI embeddings) ──► top-5 candidate types
   │
   ▼
② LLM decision ── gpt-4o-mini, JSON schema with request_type ∈ {candidates, UNCLEAR}
   │                 + confidence, location, follow_up_question
   ▼
③ lookup ──────── type ──► department, queue, in-scope flag (from Part A's table, never generated)
   │
   └─ confidence low / UNCLEAR ──► ask one follow-up ──► resident answers ──► back to ①
```

**Course techniques used:** RAG (embeddings + FAISS) · structured output (Pydantic / JSON schema, enum-constrained) · prompt engineering · LLM data enrichment (type cards).

| Input (from Part A) | Output (this notebook) |
|---|---|
| `Data/request_types.csv` — 170 de-duplicated types | `C_intake/type_cards.json` — LLM-written description + resident phrasings per type |
| `Data/intake_test_set.csv` — 45 labelled test messages (v2) | `C_intake/intake_predictions.csv`, `C_intake/intake_eval_summary.csv` |

**Before running:** add the shared `IE5374Project` folder to *My Drive* (Shared with me → right-click → Organize → Add shortcut), put `intake_test_set.csv` in `IE5374Project/Data/`, and store your key as the Colab secret `OPENAI_API_KEY` (🔑 in the left sidebar, enable notebook access).

> **AI use:** the code in this notebook and the first draft of the test set were written with an AI coding assistant (Claude) and reviewed by the team. The type descriptions in `type_cards.json` are generated by gpt-4o-mini in Step 1. Test messages are AI-drafted and checked by hand against the real type list.

## 0 · Setup

In [15]:
!pip install -q langchain-openai langchain-community faiss-cpu gradio

In [16]:
import os, re, json, html
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import pandas as pd

try:  # Colab: mount Drive and read the key from Colab secrets
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    DATA_DIR = Path("/content/drive/MyDrive/IE5374Project/Data")
except ImportError:  # local run: expects ./data and OPENAI_API_KEY in the environment
    DATA_DIR = Path("data")

OUT_DIR = DATA_DIR.parent / "C_intake"
OUT_DIR.mkdir(exist_ok=True)

CHAT_MODEL = "gpt-4o-mini"              # course DEFAULT_MINI_MODEL
EMBED_MODEL = "text-embedding-3-small"  # course DEFAULT_EMBED_MODEL
TOP_K = 5                                # candidates passed from retrieval to the LLM

IN_SCOPE = {"PWDx", "BTDT", "PARK", "ISD"}  # the four departments in the staffing plan
DEPT_NAMES = {
    "PWDx": "Public Works",
    "BTDT": "Transportation (parking & traffic)",
    "PARK": "Parks & Recreation",
    "ISD": "Inspectional Services",
    "BWSC": "Boston Water & Sewer Commission",
    "INFO": "311 general information / referral form",
    "GEN": "Needle pickup program",
    "PROP": "Property Management (graffiti removal)",
}
print("data:", DATA_DIR, "| outputs:", OUT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
data: /content/drive/MyDrive/IE5374Project/Data | outputs: /content/drive/MyDrive/IE5374Project/C_intake


In [17]:
def clean(s):
    """Unescape HTML codes like &amp; and collapse whitespace."""
    return re.sub(r"\s+", " ", html.unescape(str(s))).strip()

def norm(s):
    return clean(s).lower()

types_df = pd.read_csv(DATA_DIR / "request_types.csv")
types_df["example_titles"] = types_df["example_titles"].map(clean)
TYPE_NAMES = list(types_df["type"])                  # canonical names, exactly as in the data
TYPE_INFO = types_df.set_index("type").to_dict("index")
NORM2TYPE = {norm(t): t for t in TYPE_NAMES}
assert len(NORM2TYPE) == len(TYPE_NAMES) == 170

def dept_of(t):
    return TYPE_INFO.get(t, {}).get("department", "NONE")

print(len(TYPE_NAMES), "types")
print(types_df["department"].value_counts().to_dict())
types_df[["type", "department", "reason", "top_queue", "count", "example_titles"]].head(8)

170 types
{'ISD': 60, 'PWDx': 55, 'BTDT': 22, 'INFO': 12, 'PARK': 11, 'BWSC': 8, 'GEN': 1, 'PROP': 1}


,type,department,reason,top_queue,count,example_titles
0,Parking Enforcement,BTDT,Enforcement & Abandoned Vehicles,BTDT_Parking Enforcement,62973,Parking Enforcement
1,Improper Storage of Trash (Barrels),PWDx,Code Enforcement,PWDx_Code Enforcement,20273,Improper Storage of Trash (Barrels) | Shopping Cart
2,Requests for Street Cleaning,PWDx,Street Cleaning,PWDx_District 1C: Downtown,20128,Requests for Street Cleaning | Street Cleaning | Pick up Dead Animal
3,CE Collection,PWDx,Street Cleaning,PWDx_District 1C: Downtown,17977,CE Collection
4,Schedule a Bulk Item Pickup,PWDx,Sanitation,PWDx_Schedule a Bulk Item Pickup,11828,Schedule a Bulk Item Pickup
5,Needle Pickup,GEN,Needle Program,GEN_Needle_Pickup,10710,Needle Pickup | Needle Clean-up
6,Missed Trash/Recycling/Yard Waste/Bulk Item,PWDx,Sanitation,PWDx_Missed Trash\Recycling\Yard Waste\Bulk Item,10584,Missed Trash: District 07 | Missed Trash: District 02 | Missed Trash: District 06
7,Poor Conditions of Property,PWDx,Code Enforcement,PWDx_Code Enforcement,8643,Poor Conditions of Property | Student Move-in CE Violation | WEEDS


## 1 · Type cards: enrich the catalog with an LLM

91 of the 170 types have example titles that only repeat the type name (e.g. *Parking Enforcement*), which gives retrieval almost nothing to match a resident's wording against. We ask the LLM, in batches of 10 with a strict JSON schema, for a short description and four resident-style phrasings per type. Results are cached in `type_cards.json`, so this step only calls the API once.

In [18]:
from typing import List, Optional, Literal
from pydantic import BaseModel, Field, create_model
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

llm = ChatOpenAI(model=CHAT_MODEL, temperature=0)

class TypeCard(BaseModel):
    type: str = Field(..., description="The request type name, copied exactly")
    description: str = Field(..., description="1-2 plain-English sentences on what problem this type covers")
    resident_phrases: List[str] = Field(..., description="4 short, varied messages a Boston resident might send for this type; no street addresses")

class TypeCardBatch(BaseModel):
    cards: List[TypeCard]

card_llm = llm.with_structured_output(TypeCardBatch, method="json_schema", strict=True)

CARD_PROMPT = """You are building the intake catalog for Boston 311, the city's service request system.
For EACH request type below, write a plain-English description and 4 short messages a resident could realistically send.

Rules:
- Copy the type name exactly.
- Base the description on the department, category, queue and example case titles given.
- If a type looks internal or administrative (used by city staff, follow-ups, notifications), say so in the description.
- Make the resident phrases sound like real people: casual, varied, sometimes vague. Do not mention other request types.

Request types:
{block}"""

def type_line(t):
    r = TYPE_INFO[t]
    return (f"- type: {t} | department: {r['department']} ({DEPT_NAMES.get(r['department'], '')}) | "
            f"category: {r['reason']} | queue: {r['top_queue']} | example case titles: {r['example_titles'][:200]}")

cards_path = OUT_DIR / "type_cards.json"
cards = json.loads(cards_path.read_text(encoding="utf-8")) if cards_path.exists() else {}

for attempt in range(3):  # retry any type the model skipped or renamed
    todo = [t for t in TYPE_NAMES if t not in cards]
    if not todo:
        break
    batches = [todo[i:i + 10] for i in range(0, len(todo), 10)]
    prompts = [CARD_PROMPT.format(block="\n".join(type_line(t) for t in b)) for b in batches]
    for out in card_llm.batch(prompts, config={"max_concurrency": 5}, return_exceptions=True):
        if isinstance(out, Exception):
            print("batch failed, will retry:", repr(out)[:200])
            continue
        for c in out.cards:
            t = NORM2TYPE.get(norm(c.type))
            if t:
                cards[t] = {"description": c.description, "resident_phrases": c.resident_phrases}
    print(f"round {attempt + 1}: {len(cards)}/{len(TYPE_NAMES)} cards")

missing = [t for t in TYPE_NAMES if t not in cards]
assert not missing, f"no card for: {missing}"
cards_path.write_text(json.dumps(cards, indent=1, ensure_ascii=False), encoding="utf-8")

pd.DataFrame([{"type": t, **cards[t]} for t in ["Parking Enforcement", "CE Collection", "Rodent Activity", "Tree Emergencies"]])

,type,description,resident_phrases
0,Parking Enforcement,This type is for reporting parking violations or issues related to abandoned vehicles....,"[There's a car parked in front of my driveway again!, Can someone check on that abando..."
1,CE Collection,This type is for reporting collections related to street cleaning efforts. It is used ...,"[I saw some debris that needs to be collected, can you take care of it?, There's a pil..."
2,Rodent Activity,This request type is for reporting sightings of rodents in the area. Residents can hel...,"[I've seen a lot of rats around my neighborhood, can you do something?, There are too ..."
3,Tree Emergencies,"This type is for urgent issues involving trees, such as fallen branches or hazardous c...","[There's a huge branch down in the street, it's blocking traffic!, A tree fell in my y..."


## 2 · Retrieval (RAG)

Each type becomes one document. We build two FAISS indexes so we can measure what the enrichment buys:
- **basic**: type name, category, department, example case titles (what Part A's table already has)
- **enriched**: basic + LLM description + resident phrasings

In [19]:
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS

def card_text(t, enriched):
    r = TYPE_INFO[t]
    parts = [f"Request type: {t}",
             f"Category: {r['reason']}",
             f"Department: {DEPT_NAMES.get(r['department'], r['department'])}",
             f"Example case titles: {r['example_titles']}"]
    if enriched:
        parts.append(f"Description: {cards[t]['description']}")
        parts.append("Residents say: " + " / ".join(cards[t]["resident_phrases"]))
    return "\n".join(parts)

emb = OpenAIEmbeddings(model=EMBED_MODEL)
stores = {}
for v in ["basic", "enriched"]:
    docs = [Document(page_content=card_text(t, v == "enriched"), metadata={"type": t}) for t in TYPE_NAMES]
    stores[v] = FAISS.from_documents(docs, emb)

def retrieve(query, variant="enriched", k=TOP_K):
    return [d.metadata["type"] for d in stores[variant].similarity_search(query, k=k)]

q = "there's a big hole on Centre St near the T stop"
pd.DataFrame({"basic": retrieve(q, "basic"), "enriched": retrieve(q, "enriched")})

,basic,enriched
0,Sidewalk Cover / Manhole,Sidewalk Cover / Manhole
1,BWSC Pothole,BWSC Pothole
2,Bicycle Issues,Fire Hydrant
3,Missing Sign,Request for Pothole Repair
4,Tree in Park,Empty Litter Basket


## 3 · Decision with structured output

The LLM sees the resident message and the candidate types. After a follow-up, candidates are retrieved twice (on the answer alone and on message + answer) and merged, up to 10 types. The JSON schema is built **per call** so that `request_type` is an enum of exactly those candidates plus `UNCLEAR`: the model cannot return a type that does not exist. Department, queue and the in-scope flag are then looked up from the data, not generated.

`queue` is the type's most common queue (`top_queue`). Some Public Works queues are district-specific (e.g. *District 1C: Downtown*); routing to the district would need a location → district lookup, which we list as a next step.

In [20]:
SYSTEM_PROMPT = """You are the intake assistant for Boston 311, the city's service request system.
You receive a resident's message and a list of candidate request types from the city's real catalog.
Choose the ONE request type that best matches what the resident wants fixed or reported.

Rules:
1. Choose only from the candidates. Choose "UNCLEAR" if none of them fits or the message is not a request for a city service.
2. Judge by the underlying problem the resident describes, not by matching keywords.
3. confidence:
   - "high": one candidate clearly fits.
   - "medium": one candidate fits best but some details are missing.
   - "low": two or more candidates handled by different teams fit about equally well and the message does not say which.
4. If confidence is "low" or the type is "UNCLEAR", write ONE short, friendly follow-up question that would tell the options apart
   (for a non-311 message, say briefly what 311 can help with). Otherwise follow_up_question must be null.
5. Write follow_up_question in the same language the resident used.
6. location: copy any address, street, intersection, landmark or neighborhood the resident mentions, else null.
7. If the resident has already answered a follow-up question, use that answer and do not ask again unless it is still unclear."""

def decision_schema(options):
    return create_model(
        "IntakeDecision",
        reasoning=(str, Field(..., description="One short sentence on why this type fits")),
        request_type=(Literal[tuple(options)], Field(..., description="The chosen request type, or UNCLEAR")),
        confidence=(Literal["high", "medium", "low"], Field(..., description="See rule 3")),
        location=(Optional[str], Field(..., description="Location mentioned by the resident, or null")),
        follow_up_question=(Optional[str], Field(..., description="Only when confidence is low or type is UNCLEAR, else null")),
    )

def candidate_line(t, variant):
    if variant == "enriched":
        return f"- {t}: {cards[t]['description']}"
    r = TYPE_INFO[t]
    return f"- {t}: {r['reason']} (examples: {r['example_titles'][:120]})"

def build_query(message, clarification=None):
    if clarification:
        return f"{message}\n(Resident's answer to the follow-up question: {clarification})"
    return message

def candidates_for(message, clarification=None, variant="enriched"):
    """Top-k for the message. After a follow-up, also retrieve on the answer alone and merge,
    so the vague first message can't drown out the detail in the answer."""
    if not clarification:
        return retrieve(message, variant)
    merged = []
    for q in [clarification, build_query(message, clarification)]:
        for t in retrieve(q, variant):
            if t not in merged:
                merged.append(t)
    return merged

def classify(message, clarification=None, mode="rag", variant="enriched"):
    """mode='rag': retrieved candidates; mode='full': all 170 types in the prompt."""
    query = build_query(message, clarification)
    cands = candidates_for(message, clarification, variant) if mode == "rag" else list(TYPE_NAMES)
    chooser = llm.with_structured_output(decision_schema(cands + ["UNCLEAR"]), method="json_schema", strict=True)
    d = chooser.invoke([
        ("system", SYSTEM_PROMPT),
        ("human", f"Resident message:\n{query}\n\nCandidate request types:\n" +
                  "\n".join(candidate_line(t, variant) for t in cands)),
    ])
    t = d.request_type
    dept = TYPE_INFO[t]["department"] if t in TYPE_INFO else None
    needs_follow_up = (t == "UNCLEAR" or d.confidence == "low") and clarification is None
    question = d.follow_up_question or "Could you tell me a bit more about the problem and exactly where it is?"
    return {
        "request_type": t,
        "department": dept,
        "department_name": DEPT_NAMES.get(dept),
        "queue": TYPE_INFO[t]["top_queue"] if t in TYPE_INFO else None,
        "in_scope": (dept in IN_SCOPE) if dept else None,
        "confidence": d.confidence,
        "location": d.location,
        "needs_follow_up": needs_follow_up,
        "follow_up_question": question if needs_follow_up else None,
        "reasoning": d.reasoning,
        "candidates": cands if mode == "rag" else None,
    }

def embed_top1(message, clarification=None, variant="basic"):
    """Baseline without an LLM: the nearest type card wins, never asks a follow-up."""
    cands = retrieve(build_query(message, clarification), variant)
    t = cands[0]
    return {"request_type": t, "department": dept_of(t), "needs_follow_up": False, "candidates": cands}

for m in ["there's a big hole on Centre St near the T stop",
          "Found a used needle in the grass at the park near my kid's school"]:
    print(m)
    print(json.dumps({k: v for k, v in classify(m).items() if k != "candidates"}, indent=1, ensure_ascii=False), "\n")

there's a big hole on Centre St near the T stop
{
 "request_type": "Request for Pothole Repair",
 "department": "PWDx",
 "department_name": "Public Works",
 "queue": "PWDx_Roadway Repair_ARP_Resurfacing",
 "in_scope": true,
 "confidence": "high",
 "location": "Centre St near the T stop",
 "needs_follow_up": false,
 "follow_up_question": null,
 "reasoning": "The resident is reporting a hole on the street, which is best categorized as a pothole issue."
} 

Found a used needle in the grass at the park near my kid's school
{
 "request_type": "Needle Pickup",
 "department": "GEN",
 "department_name": "Needle pickup program",
 "queue": "GEN_Needle_Pickup",
 "in_scope": false,
 "confidence": "high",
 "location": null,
 "needs_follow_up": false,
 "follow_up_question": null,
 "reasoning": "The resident is reporting a used needle found in the grass, which directly relates to the need for needle pickups in public areas."
} 



## 4 · Follow-up conversation

In [21]:
msg = "There's trash everywhere on my street."
first = classify(msg)
print("Resident :", msg)
print("Assistant:", first["follow_up_question"] or f"Filed as {first['request_type']}")

answer = "Someone dumped a whole pile of furniture and garbage bags on the corner last night."
second = classify(msg, clarification=answer)
print("Resident :", answer)
print("Assistant: filed as", second["request_type"], "→", second["department_name"], "/", second["queue"],
      f"(confidence {second['confidence']})")

Resident : There's trash everywhere on my street.
Assistant: Could you clarify if the trash is coming from a dumpster, litter basket, or if it's just scattered on the street?
Resident : Someone dumped a whole pile of furniture and garbage bags on the corner last night.
Assistant: filed as Illegal Dumping → Public Works / PWDx_Code Enforcement (confidence high)


## 5 · Evaluation on the test set

`intake_test_set.csv` (v2) has 45 resident messages:

| category | n | what counts as correct |
|---|---|---|
| clear | 30 | predicted type is in `acceptable_types` (some messages have 2–3 equally valid types) |
| ambiguous | 9 | the assistant asks a follow-up; after the scripted `clarification`, the type is acceptable |
| other_dept | 5 | correct type outside the four departments (BWSC, GEN, INFO, PROP), flagged `in_scope = False` |
| not_311 | 1 | returns `UNCLEAR` and asks instead of forcing a type |

End-to-end rule: if the assistant asks a follow-up and the row has a clarification, the answer after the clarification is scored; otherwise the first answer is scored. Three ambiguous messages appear twice with different clarifications (T30/T42, T31/T43, T33/T44), so an assistant that guesses instead of asking can get at most one of each pair right.

We compare five set-ups to show what each component adds.

#### Results log: v1 → v2
**v1** (41 messages, written before any prompt work; file kept as `intake_test_set_v1.csv`):

| set-up | type acc (all) | type acc (clear) | dept acc | retrieval recall | asks when it should | asks when it shouldn't |
|---|---|---|---|---|---|---|
| A · embedding top-1 (basic) | 54% | 52% | 83% | 88% | 0% | 0% |
| B · embedding top-1 (enriched) | 66% | 69% | 85% | 95% | 0% | 0% |
| C · RAG + LLM (basic) | 83% | 86% | 93% | 88% | 43% | 0% |
| D · RAG + LLM (enriched) ★ | 95% | 97% | 100% | 98% | 71% | 0% |
| E · all 170 types + LLM | 98% | 100% | 100% | n/a | 14% | 0% |

**What changed after reviewing the v1 errors of set-up D** (the system prompt was not changed):
1. *Bug fix:* after a follow-up, retrieval used message + answer as one query and the vague message drowned out the answer (T31 lost the right type from the candidates). Now it also retrieves on the answer alone and merges.
2. *Label fixes:* T24 also accepts *Food Alert - Unconfirmed* (its real case titles include "Unsanitary Conditions - Food"; same department and queue). T34 moved from ambiguous to clear: all its candidates route to ISD Housing, so by rule 3 no follow-up is needed.
3. *Harder ambiguous cases:* in v1 every clarification pointed to the most common reading, so guessing scored as well as asking (set-up E asked only 14% of the time and still scored 98%). v2 adds T42–T44, the same messages with clarifications pointing to the less common reading, and T45, a new cross-team ISD case.

In [22]:
test = pd.read_csv(DATA_DIR / "intake_test_set.csv").fillna("")
test["acceptable"] = test["acceptable_types"].str.split("|")
unknown = {t for ts in test["acceptable"] for t in ts if t != "UNCLEAR" and t not in TYPE_INFO}
assert not unknown, f"labels not in the catalog: {unknown}"
print(test["category"].value_counts().to_dict())
test[["id", "category", "message", "gold_type", "clarification"]].head(8)

{'clear': 30, 'ambiguous': 9, 'other_dept': 5, 'not_311': 1}


,id,category,message,gold_type,clarification
0,T01,clear,"pothole on centre st right by the jackson sq T stop, almost wrecked my tire!!",Request for Pothole Repair,
1,T02,clear,我家楼下的路灯坏了一个星期了，晚上很黑 (in front of 42 Bowdoin St),Street Light Outages,
2,T03,clear,"No recogieron la basura en mi calle este martes, todas las bolsas siguen en la acera. ...",Missed Trash/Recycling/Yard Waste/Bulk Item,
3,T04,clear,I need to get rid of an old couch and a dresser. How do I schedule a pickup?,Schedule a Bulk Item Pickup,
4,T05,clear,Can someone come take away my old mattress? It's wrapped in plastic on the curb.,Mattress_Pickup,
5,T06,clear,My neighbors leave open trash bags on the sidewalk all week instead of using barrels.,Improper Storage of Trash (Barrels),
6,T07,clear,Someone dumped a pile of old tires and junk in the empty lot on Blue Hill Ave overnight.,Illegal Dumping,
7,T08,clear,"The store on the corner never shoveled their sidewalk, it's a sheet of ice and my mom ...",Unshoveled Sidewalk,


In [23]:
def score_row(row, first, second=None):
    """first = answer to the original message; second = answer after the clarification (if one was used)."""
    ok = set(row["acceptable"])
    final = second if second is not None else first
    cands = final.get("candidates")
    return {
        "id": row["id"],
        "category": row["category"],
        "pred_first": first["request_type"],
        "pred_final": final["request_type"],
        "acceptable": "|".join(row["acceptable"]),
        "retrieval_hit": (bool(ok & set(cands)) if cands is not None and "UNCLEAR" not in ok else None),
        "type_correct": final["request_type"] in ok,
        "dept_correct": dept_of(final["request_type"]) in {dept_of(t) for t in ok},
        "asked_follow_up": bool(first["needs_follow_up"]),
        "expected_follow_up": bool(int(row["expected_follow_up"])),
        "candidates": "|".join(cands) if cands is not None else "",
    }

def summarize(scored):
    s = pd.DataFrame(scored)
    clear = s[s.category == "clear"]
    no_ask_expected = s[~s.expected_follow_up]
    ask_expected = s[s.expected_follow_up]
    hits = s["retrieval_hit"].dropna().astype(float)
    return {
        "type accuracy (all)": s.type_correct.mean(),
        "type accuracy (clear)": clear.type_correct.mean(),
        "department accuracy (all)": s.dept_correct.mean(),
        "retrieval recall": hits.mean() if len(hits) else float("nan"),  # top-5, up to 10 after a follow-up
        "asks when it should": ask_expected.asked_follow_up.mean(),
        "asks when it shouldn't": no_ask_expected.asked_follow_up.mean(),
    }

In [24]:
SETUPS = {
    "A · embedding top-1 (basic cards)":    lambda m, c=None: embed_top1(m, c, variant="basic"),
    "B · embedding top-1 (enriched cards)": lambda m, c=None: embed_top1(m, c, variant="enriched"),
    "C · RAG top-5 + LLM (basic cards)":    lambda m, c=None: classify(m, c, mode="rag", variant="basic"),
    "D · RAG top-5 + LLM (enriched) ★":     lambda m, c=None: classify(m, c, mode="rag", variant="enriched"),
    "E · all 170 types in prompt + LLM":    lambda m, c=None: classify(m, c, mode="full", variant="enriched"),
}

def run_row(predict, row):
    first = predict(row["message"])
    second = predict(row["message"], row["clarification"]) if (first["needs_follow_up"] and row["clarification"]) else None
    return score_row(row, first, second)

results, summary = {}, {}
rows = [r for _, r in test.iterrows()]
for name, predict in SETUPS.items():
    with ThreadPoolExecutor(max_workers=8) as pool:
        results[name] = list(pool.map(lambda r: run_row(predict, r), rows))
    summary[name] = summarize(results[name])
    print(f"done: {name}")

summary_df = pd.DataFrame(summary).T
summary_df.to_csv(OUT_DIR / "intake_eval_summary.csv")
pd.concat({k: pd.DataFrame(v) for k, v in results.items()}, names=["setup"]).to_csv(OUT_DIR / "intake_predictions.csv")
summary_df.style.format("{:.0%}", na_rep="n/a")

done: A · embedding top-1 (basic cards)
done: B · embedding top-1 (enriched cards)
done: C · RAG top-5 + LLM (basic cards)
done: D · RAG top-5 + LLM (enriched) ★
done: E · all 170 types in prompt + LLM


,type accuracy (all),type accuracy (clear),department accuracy (all),retrieval recall,asks when it should,asks when it shouldn't
A · embedding top-1 (basic cards),49%,53%,80%,80%,0%,0%
B · embedding top-1 (enriched cards),60%,70%,82%,89%,0%,0%
C · RAG top-5 + LLM (basic cards),76%,87%,89%,82%,40%,0%
D · RAG top-5 + LLM (enriched) ★,98%,100%,98%,98%,70%,0%
E · all 170 types in prompt + LLM,89%,100%,98%,n/a,10%,0%


### Error analysis (main set-up D)

In [25]:
main = pd.DataFrame(results["D · RAG top-5 + LLM (enriched) ★"]).merge(test[["id", "message", "clarification"]], on="id")
errors = main[~main.type_correct | (main.asked_follow_up != main.expected_follow_up)]
pd.set_option("display.max_colwidth", 90)
errors[["id", "category", "message", "pred_first", "pred_final", "acceptable", "asked_follow_up", "retrieval_hit", "candidates"]]

,id,category,message,pred_first,pred_final,acceptable,asked_follow_up,retrieval_hit,candidates
29,T30,ambiguous,The light on my corner is broken.,Street Light Outages,Street Light Outages,Street Light Outages,False,True,Knockdown Replacement|Street Light Knock Downs|Street Light Outages|General Lighting R...
41,T42,ambiguous,The light on my corner is broken.,Street Light Outages,Street Light Outages,Traffic Signal Inspection|Traffic Signal Repair,False,False,Knockdown Replacement|Street Light Knock Downs|Street Light Outages|General Lighting R...
42,T43,ambiguous,There's trash everywhere on my street.,Illegal Dumping,Illegal Dumping,Illegal Dumping,False,True,Trash on Vacant Lot|Empty Litter Basket|Improper Storage of Trash (Barrels)|Overflowin...


## 6 · Demo app (Gradio)

Type a message as a resident. If the assistant asks a follow-up, your next message is treated as the answer.

In [26]:
import gradio as gr

def ticket_markdown(res):
    scope = "in the 2025 staffing plan" if res["in_scope"] else "handled outside the four planned departments"
    return (f"**Request type:** {res['request_type']}  \n"
            f"**Department:** {res['department_name']} ({res['department']}) — {scope}  \n"
            f"**Queue:** {res['queue']}  \n"
            f"**Location:** {res['location'] or 'not given'}  \n"
            f"**Confidence:** {res['confidence']}")

def respond(message, chat, pending):
    if pending:                      # this message answers our follow-up question
        res = classify(pending, clarification=message)
    else:
        res = classify(message)
    if res["needs_follow_up"]:
        reply, pending = res["follow_up_question"], message
    elif res["request_type"] == "UNCLEAR":
        reply, pending = "Sorry, I couldn't match this to a 311 service request. A 311 agent will follow up.", None
    else:
        reply, pending = ticket_markdown(res), None
    chat = (chat or []) + [{"role": "user", "content": message}, {"role": "assistant", "content": reply}]
    shown = {k: v for k, v in res.items() if k != "candidates"}
    return chat, pending, shown, ""

with gr.Blocks() as demo:
    gr.Markdown("## Boston 311 Intake Assistant\nDescribe a problem in your neighborhood.")
    try:
        chatbot = gr.Chatbot(type="messages", height=380)
    except TypeError:                # newer Gradio: messages format is the default
        chatbot = gr.Chatbot(height=380)
    pending = gr.State(None)
    box = gr.Textbox(placeholder="e.g. there's a big hole on Centre St near the T stop", label="Your message")
    raw = gr.JSON(label="Structured output")
    box.submit(respond, [box, chatbot, pending], [chatbot, pending, raw, box])
    gr.Button("New conversation").click(lambda: ([], None, None, ""), None, [chatbot, pending, raw, box])

# share=False keeps the app inside this Colab session. A share=True link is public for up to a week and
# anyone with it spends your OpenAI credit; if you need it for recording, run demo.close() afterwards.
demo.launch(share=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

## 7 · Notes for the report

- **Why RAG + enum-constrained output:** retrieval keeps the prompt short and focused; the per-call enum makes invalid types impossible, and department/queue come from the data, so routing can't be hallucinated. Compare set-ups D and E in the summary table to see whether retrieval costs or buys accuracy versus putting all 170 types in the prompt.
- **What enrichment buys:** compare A→B (retrieval only) and C→D (with the LLM).
- **Asking vs guessing:** v2's paired ambiguous messages (T30/T42, T31/T43, T33/T44) show how much the follow-up question is worth; compare D and E on "asks when it should" and type accuracy.
- **Limitations:** the test set is small (45) and AI-drafted, and v2 was revised after seeing v1 errors (see the results log in Section 5); ideally teammates add ~10 messages written blind. `queue` is the most common queue for the type; district-level Public Works queues need a location → district lookup. 42% of real requests are logged by city employees, whose wording differs from residents'.
- **Next steps:** district routing from `location`; classify a batch of messages and sum by department to feed Part B's monthly load.